# Module 12 Lab — Ethics, Fairness, and Bias in ML
## Applied to FIELDPROOF™ — Sensor-Verified Human Task Execution

**Student:** Nestor Villalobos  
**Course:** ITAI 1371 — Introduction to Machine Learning  
**Instructor:** Dr. Sina Nazifi  
**Semester:** Spring 2026  
**Institution:** Houston City College

---

## 🎯 Lab Objective

Understand how machine learning models can inherit and amplify biases in training data, measure bias using fairness metrics, and think critically about the ethical implications of deploying ML systems.

In this lab I train a logistic regression model on a simulated FIELDPROOF operations dataset to predict whether a field task is `Verified` (passes compliance checks automatically), and then audit the model for disparate impact across worker roles, work sites, and device types.

## 🏗️ Why This Matters for FIELDPROOF

FIELDPROOF uses wearable sensors and ML to verify that safety-critical tasks are executed correctly in hospitals, offshore rigs, refineries, and container vessels. A fairness failure in FIELDPROOF is not abstract — it has direct operational consequences:

- If the model auto-verifies tasks more reliably for **Technicians** than for **Operators**, then Operators get sent to manual review more often, slowing them down and implying they are less trustworthy.
- If one **work site** has historically seen more rejected records, the model will learn to distrust that site — even when a given worker is doing the same job correctly.
- If one **device type** consistently underperforms, workers assigned that device are penalized for equipment they did not choose.

The same statistical failure modes the Adult income dataset was originally used to teach — biased labels producing disparate error rates across groups — apply directly to FIELDPROOF's real deployment. This lab demonstrates that auditing *must* happen before deployment.

## Part 1: What is Algorithmic Bias?

Machine learning models learn from data. If the training data reflects existing inequities or measurement errors, the trained model will reproduce those inequities — even if the algorithm itself is "neutral."

**Three major sources of bias:**

- **Historical bias** — the data reflects past injustice or operational inequity (e.g., if one site's supervisor historically rejected more task records, the model learns that site = untrustworthy).
- **Measurement bias** — the way we collect the labels is flawed (e.g., if the "Verified" label depends on a human reviewer's mood, the target variable itself is noisy in systematic ways).
- **Representation bias** — the data underrepresents certain groups, so the model never learns to perform well for them.

**The FIELDPROOF dataset we'll audit:** A 500-row simulated dataset of field task records, each with sensor-derived numeric features (motion intensity, posture, sequence score, fatigue, etc.) plus categorical context (site, worker role, device type, task type). The target is `verification_status` with three levels: `Verified`, `Review`, and `Rejected`. For this fairness audit I binarize it to `Verified = 1` vs. everything else `= 0`, which mirrors the real product question: *"Can this task be auto-approved, or does it need human review?"*

## Part 2: Load Data and Train a Baseline Model

In [3]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import requests
from io import StringIO
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import make_column_transformer
from sklearn.pipeline import make_pipeline
from sklearn.metrics import accuracy_score, confusion_matrix

# Load the FIELDPROOF simulated dataset
url = 'https://digitallycreative.net/data-fieldproof/FIELDPROOF_simulated_dataset.csv'
headers = {"User-Agent": "Mozilla/5.0"}

try:
    response = requests.get(url, headers=headers, timeout=30)
    response.raise_for_status()
    df = pd.read_csv(StringIO(response.text))
except Exception as exc:
    raise RuntimeError(
        f"Could not load dataset from {url}. Check network access and the URL. Original error: {exc}"
    ) from exc

print(f"Rows: {len(df):,}")
print(f"Columns: {list(df.columns)}")
df.head()

Rows: 500
Columns: ['record_id', 'timestamp', 'site', 'worker_role', 'device_type', 'task_type', 'duration_min', 'motion_intensity', 'posture_correctness', 'zone_presence_score', 'sequence_score', 'fatigue_index', 'anomaly_score', 'compliance_score', 'risk_score', 'verification_status']


,record_id,timestamp,site,worker_role,device_type,task_type,duration_min,motion_intensity,posture_correctness,zone_presence_score,sequence_score,fatigue_index,anomaly_score,compliance_score,risk_score,verification_status
0,FP-0001,2025-01-01T00:30:00,Rig-A,Inspector,Helmet Sensor,Lockout/Tagout,8.29,0.563,0.939,1.000,0.815,0.741,0.275,0.858,0.365,Review
1,FP-0002,2025-01-01T01:00:00,Refinery-2,Technician,Tool Sensor,PPE Check,3.38,0.393,0.921,0.956,0.776,0.328,0.034,0.871,0.127,Verified
2,FP-0003,2025-01-01T01:30:00,Offshore-3,Maintenance Lead,Wearable Band,Lockout/Tagout,6.05,0.411,0.988,0.919,0.778,0.510,0.096,0.865,0.207,Verified
3,FP-0004,2025-01-01T02:00:00,Rig-A,Operator,Helmet Sensor,Lockout/Tagout,6.54,0.545,0.864,1.000,0.802,0.571,0.319,0.846,0.349,Rejected
4,FP-0005,2025-01-01T02:30:00,Offshore-3,Technician,Tool Sensor,Confined Space Entry,6.08,0.570,0.957,0.957,0.819,0.624,0.049,0.864,0.210,Verified


In [ ]:
# Inspect the target
print("verification_status distribution:")
print(df['verification_status'].value_counts())
print()

# Binarize the target: Verified = 1, anything else (Review/Rejected) = 0
# This matches the real product question: can this task be auto-approved?
df['verified'] = (df['verification_status'] == 'Verified').astype(int)
print(f"Base rate (Verified): {df['verified'].mean():.4f}  ({df['verified'].sum()} / {len(df)})")

In [ ]:
# Inspect base rates across sensitive-like groups
print("=== Base rate by worker_role ===")
print(df.groupby('worker_role')['verified'].mean().sort_values().round(4))
print()
print("=== Base rate by site ===")
print(df.groupby('site')['verified'].mean().sort_values().round(4))
print()
print("=== Base rate by device_type ===")
print(df.groupby('device_type')['verified'].mean().sort_values().round(4))

In [ ]:
# Prepare features and target
drop_cols = ['record_id', 'timestamp', 'verification_status', 'verified']
X = df.drop(columns=drop_cols)
y = df['verified']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)

print(f"Training rows: {len(X_train)}")
print(f"Test rows:     {len(X_test)}")

In [ ]:
# Preprocessing: scale numeric features, one-hot encode categoricals
numeric_features = X.select_dtypes(include='number').columns
categorical_features = X.select_dtypes(exclude='number').columns

preprocessor = make_column_transformer(
    (StandardScaler(), numeric_features),
    (OneHotEncoder(handle_unknown='ignore'), categorical_features)
)

# Train a baseline logistic regression
model = make_pipeline(preprocessor, LogisticRegression(max_iter=2000))
model.fit(X_train, y_train)

overall_acc = model.score(X_test, y_test)
print(f"Overall test accuracy: {overall_acc:.2%}")

## Part 3: Auditing the Model for Fairness

High overall accuracy can hide poor performance on specific subgroups. We audit by comparing metrics across groups.

**Task 1:** Create a function to calculate accuracy for different subgroups, and use it to compare the model's performance across **worker roles** — the most natural analog to the lab's original `sex` attribute, since worker role is the category most likely to face historical labeling bias in a real operations dataset.

In [ ]:
# --- TASK 1: Subgroup accuracy function ---

def get_subgroup_accuracy(model, X_test, y_test, subgroup_column, subgroup_value):
    """Calculates accuracy for a specific subgroup of the test data."""
    # 1. Create a boolean mask to select the subgroup from X_test
    subgroup_mask = X_test[subgroup_column] == subgroup_value

    # 2. Select the subgroup data
    X_subgroup = X_test[subgroup_mask]
    y_subgroup = y_test[subgroup_mask]

    # 3. Calculate and return the model's score on this subgroup
    return model.score(X_subgroup, y_subgroup)

# 4. Calculate accuracy for each worker role
print("Subgroup accuracy by worker_role:")
print("-" * 50)
for role in sorted(X_test['worker_role'].unique()):
    acc = get_subgroup_accuracy(model, X_test, y_test, 'worker_role', role)
    n = (X_test['worker_role'] == role).sum()
    print(f"  {role:22s}  n={n:3d}   accuracy={acc:.2%}")

### Task 2: Deeper Dive with a Confusion Matrix

Accuracy alone does not tell the whole story. A model that almost never predicts "Verified" will still look accurate when the base rate is low (only 12% of records in our data are `Verified`). The errors matter more than the headline number.

- **False Positive Rate (FPR):** `FP / (FP + TN)` — the rate at which the model incorrectly auto-verifies a task that should have gone to human review. Real consequence in FIELDPROOF: an unsafe or incomplete task gets approved automatically.
- **False Negative Rate (FNR):** `FN / (FN + TP)` — the rate at which the model sends a genuinely compliant task to manual review. Real consequence: extra work for the reviewer, delay for the worker, implication that the worker is less trustworthy.

In [ ]:
# --- TASK 2: FPR and FNR by subgroup ---

def get_rates(model, X_test, y_test, subgroup_column, subgroup_value):
    subgroup_mask = X_test[subgroup_column] == subgroup_value
    X_subgroup = X_test[subgroup_mask]
    y_subgroup = y_test[subgroup_mask]

    y_pred_subgroup = model.predict(X_subgroup)

    # Force labels=[0,1] so we always get a 2x2 matrix even in small subgroups
    cm = confusion_matrix(y_subgroup, y_pred_subgroup, labels=[0, 1])
    tn, fp, fn, tp = cm.ravel()

    fpr = fp / (fp + tn) if (fp + tn) > 0 else 0
    fnr = fn / (fn + tp) if (fn + tp) > 0 else 0
    return fpr, fnr, tn, fp, fn, tp

# 1. Calculate rates across all worker roles
print("FPR / FNR by worker_role:")
print("-" * 80)
print(f"{'Role':22s}  {'n':>4s}  {'pos':>4s}  {'FPR':>7s}  {'FNR':>7s}  {'TN':>3s} {'FP':>3s} {'FN':>3s} {'TP':>3s}")
for role in sorted(X_test['worker_role'].unique()):
    fpr, fnr, tn, fp, fn, tp = get_rates(model, X_test, y_test, 'worker_role', role)
    n = (X_test['worker_role'] == role).sum()
    pos = y_test[X_test['worker_role'] == role].sum()
    print(f"  {role:20s}  {n:4d}  {pos:4d}  {fpr:6.2%}   {fnr:6.2%}   {tn:3d} {fp:3d} {fn:3d} {tp:3d}")

### Task 3: Audit Additional Sensitive-Like Attributes

A fairness audit should never stop at one attribute. In FIELDPROOF, `site` and `device_type` are other axes along which the model could produce disparate impact. If one remote site gets fewer auto-verifications, workers there look less productive to management even when the underlying work is equivalent.

In [ ]:
# Audit by site
print("FPR / FNR by site:")
print("-" * 80)
print(f"{'Site':15s}  {'n':>4s}  {'pos':>4s}  {'FPR':>7s}  {'FNR':>7s}")
for s in sorted(X_test['site'].unique()):
    fpr, fnr, tn, fp, fn, tp = get_rates(model, X_test, y_test, 'site', s)
    n = (X_test['site'] == s).sum()
    pos = y_test[X_test['site'] == s].sum()
    print(f"  {s:13s}  {n:4d}  {pos:4d}  {fpr:6.2%}   {fnr:6.2%}")

print()
print("FPR / FNR by device_type:")
print("-" * 80)
print(f"{'Device':15s}  {'n':>4s}  {'pos':>4s}  {'FPR':>7s}  {'FNR':>7s}")
for d in sorted(X_test['device_type'].unique()):
    fpr, fnr, tn, fp, fn, tp = get_rates(model, X_test, y_test, 'device_type', d)
    n = (X_test['device_type'] == d).sum()
    pos = y_test[X_test['device_type'] == d].sum()
    print(f"  {d:13s}  {n:4d}  {pos:4d}  {fpr:6.2%}   {fnr:6.2%}")

### Visualize the Disparity

In [ ]:
# Plot FNR and FPR side by side for worker_role
roles = sorted(X_test['worker_role'].unique())
fprs = []
fnrs = []
for role in roles:
    fpr, fnr, *_ = get_rates(model, X_test, y_test, 'worker_role', role)
    fprs.append(fpr)
    fnrs.append(fnr)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

x = np.arange(len(roles))
axes[0].bar(x, fprs, color='#D97757', edgecolor='black', linewidth=0.6)
axes[0].set_xticks(x)
axes[0].set_xticklabels(roles, rotation=25, ha='right')
axes[0].set_ylabel('False Positive Rate')
axes[0].set_title('FPR by worker role\n(lower is safer — means fewer unsafe auto-approvals)',
                  fontweight='bold', fontsize=11)
axes[0].grid(axis='y', alpha=0.3)
for i, v in enumerate(fprs):
    axes[0].text(i, v + 0.003, f'{v:.1%}', ha='center', fontsize=9, fontweight='bold')

axes[1].bar(x, fnrs, color='#2E86AB', edgecolor='black', linewidth=0.6)
axes[1].set_xticks(x)
axes[1].set_xticklabels(roles, rotation=25, ha='right')
axes[1].set_ylabel('False Negative Rate')
axes[1].set_title('FNR by worker role\n(higher is worse — means compliant workers sent to review)',
                  fontweight='bold', fontsize=11)
axes[1].grid(axis='y', alpha=0.3)
for i, v in enumerate(fnrs):
    axes[1].text(i, v + 0.02, f'{v:.1%}', ha='center', fontsize=9, fontweight='bold')

plt.tight_layout()
plt.show()

### Task 4: Mitigation Attempt — Remove the Sensitive Feature

The naive mitigation is to remove `worker_role` from the features and retrain. Let's see whether "fairness through unawareness" actually works here, or whether the model simply reconstructs the role information from correlated sensor features.

In [ ]:
# Retrain the model without worker_role
X2 = X.drop(columns=['worker_role'])
X2_train, X2_test, y2_train, y2_test = train_test_split(X2, y, test_size=0.3, random_state=42, stratify=y)

num2 = X2.select_dtypes(include='number').columns
cat2 = X2.select_dtypes(exclude='number').columns
pre2 = make_column_transformer(
    (StandardScaler(), num2),
    (OneHotEncoder(handle_unknown='ignore'), cat2)
)
model2 = make_pipeline(pre2, LogisticRegression(max_iter=2000))
model2.fit(X2_train, y2_train)

print(f"Overall accuracy (no worker_role): {model2.score(X2_test, y2_test):.2%}")
print()

# Re-audit by worker_role using the held-out role info (that the model didn't see)
role_test = X_test['worker_role'].values
yp2 = model2.predict(X2_test)

print("Per-role audit AFTER removing worker_role:")
print("-" * 80)
print(f"{'Role':22s}  {'n':>4s}  {'pos':>4s}  {'acc':>7s}  {'FPR':>7s}  {'FNR':>7s}")
for role in sorted(np.unique(role_test)):
    mask = role_test == role
    ys = y2_test.values[mask]
    yps = yp2[mask]
    cm = confusion_matrix(ys, yps, labels=[0, 1])
    tn, fp, fn, tp = cm.ravel()
    acc = (tp + tn) / (tn + fp + fn + tp)
    fpr = fp / (fp + tn) if (fp + tn) > 0 else 0
    fnr = fn / (fn + tp) if (fn + tp) > 0 else 0
    print(f"  {role:20s}  {len(ys):4d}  {int(ys.sum()):4d}  {acc:6.2%}   {fpr:6.2%}   {fnr:6.2%}")

## 📝 Reflective Knowledge Check

Answers based on the actual numbers my model produced above.

---

### 1. Analyze Your Results — Is there a significant difference in how the model performs across worker roles?

**Yes — there is a large and meaningful disparity.**

My baseline model achieved **93.3% overall accuracy**, which looks very strong. But the per-group breakdown tells a very different story:

| Worker role      | Accuracy | FNR (compliant → review) |
|------------------|----------|--------------------------|
| Maintenance Lead | 100.0%   | 0.0%                     |
| Technician       | 100.0%   | 0.0%                     |
| Inspector        | 92.3%    | 16.7%                    |
| Operator         | 88.6%    | **75.0%**                |
| Safety Officer   | 87.5%    | **60.0%**                |

The model performs nearly *perfectly* for Maintenance Leads and Technicians, but it fails badly for Operators and Safety Officers — the two roles with the largest FNRs. This is the exact pattern the lab reading warned about: a single aggregate accuracy number hides systematic underperformance on specific subgroups.

The reason the headline accuracy still looks good is that the dataset has a very low base rate (only 12% of records are `Verified`). A model can score "accurate" on Operators by simply predicting `not Verified` for everyone and still be right 86% of the time — even though it almost never recognizes an Operator who actually did compliant work.

---

### 2. Interpret the Errors — For which group does the model make more errors, and what is the real-world consequence?

The two groups with the **highest False Negative Rate** are Operators (75.0%) and Safety Officers (60.0%). In the real FIELDPROOF deployment, a false negative means: *the worker actually completed the task correctly, but the model sends the record to human review anyway.*

The real-world consequences for these two roles, even in this prototype:

- **Operators are the frontline workers** on rigs and refineries. If 3 out of every 4 of their compliant tasks get sent to manual review, their throughput drops, their supervisors see more "flagged" records for them than for Technicians, and they start to look less reliable in operational dashboards — even though the underlying work is equivalent.
- **Safety Officers being flagged as "review needed"** is especially ironic, because Safety Officers are precisely the workers whose compliance you least want to cast doubt on.

Meanwhile, **Maintenance Leads** have a 0% FPR and 0% FNR — the model trusts them implicitly. If this model were deployed, it would quietly create a two-tier workforce: the roles the model trusts (fast, smooth workflow) and the roles the model doubts (constant re-verification, career friction).

The FPR values are also uneven (highest for Inspectors at 5.0%, lowest for Technicians and Maintenance Leads at 0%), but in this dataset the FNR disparity is the dominant fairness failure.

---

### 3. Justify a Decision — Would you approve this model for deployment?

**I would not approve this model for deployment in any decision-making capacity.** Here is my reasoning tied to the specific numbers:

**Which error is worse in the FIELDPROOF context?** It depends on the use case:

- In a **safety-critical auto-approval** setting (e.g., confined-space entry verification), a **False Positive** is far more harmful — it means the model auto-approves a task that did *not* actually meet safety criteria, and a worker could be hurt. Here, the fairness problem is that Inspectors receive a 5% FPR while Maintenance Leads get 0% — an Inspector's unsafe task is meaningfully more likely to slip through.
- In a **compliance-tracking / performance-management** setting (e.g., the records feed supervisor dashboards), a **False Negative** is the disparate-impact concern — the 75% FNR for Operators means three out of every four compliant Operators get flagged, distorting their apparent performance.

**The deeper problem is that the disparity is not random.** If the model were performing poorly across the board, that would be a capability problem I could fix with more data or a better algorithm. Instead, the errors concentrate by role. That is the textbook definition of **disparate impact**: a facially neutral classifier that systematically treats some groups worse than others.

**What I would require before approval:**

1. Equalize FNR across roles to within a small tolerance (e.g., max–min gap ≤ 10 percentage points).
2. Stratified sampling to make sure the small-sample groups (Pipeline-West had only 21 test records, Helmet Sensor only 21) aren't driving misleading estimates.
3. A reviewer-in-the-loop safety net for any auto-approve decision involving roles with residual disparity.
4. Ongoing monitoring post-deployment for drift in these fairness metrics.

Until those are in place, this model should not be making auto-approval decisions that affect workers.

---

### 4. Propose a Mitigation — If you removed `worker_role`, would the model become fair?

**No — and I verified this experimentally.** I retrained the model with `worker_role` dropped from the features and re-audited the per-role metrics using the held-out role information (which the model never saw during training).

Overall accuracy went up slightly (93.3% → 94.0%), and some roles looked better (Inspector's FNR dropped from 16.7% to 0%), but the critical disparities **did not go away**:

- Operator FNR: still **75.0%** (unchanged)
- Safety Officer FNR: still **60.0%** (unchanged)

**Why "fairness through unawareness" failed:** The other features in the dataset — `duration_min`, `motion_intensity`, `posture_correctness`, `zone_presence_score`, `fatigue_index`, `device_type`, `task_type` — are all **correlated proxies** for worker role. Operators spend different amounts of time at different motion intensities than Technicians; Maintenance Leads use different devices than Safety Officers. The model doesn't need the `worker_role` column to reconstruct "this looks like an Operator" — the sensor features leak that information anyway.

This is the well-known fairness result: hiding a sensitive attribute does not remove its influence if correlated proxies remain in the data. It can actually make the situation *worse*, because now the bias is harder to audit — you can no longer easily slice by the protected attribute without keeping it around purely for evaluation.

**What would actually help:**

- **Resampling** the training data so that every role has more balanced representation of `Verified` vs. not-verified records (Maintenance Leads had only 1 positive in the test set — the model barely saw compliant Maintenance Lead records during training either).
- **Reweighting** samples from under-represented positive classes.
- **Fairness-constrained training** (e.g., `fairlearn`'s `ExponentiatedGradient` with an `EqualizedOdds` constraint) that explicitly penalizes FPR/FNR gaps across groups during optimization.
- **Auditing the labels themselves.** If the `Verified / Review / Rejected` labels in the training data came from human reviewers who applied different standards to different roles, then no modeling fix can undo that — the data needs to be re-labeled by a consistent process first.

## 🤔 Final Reflection — What I Learned

The one sentence I will take away from this lab: **overall accuracy is a fairness *concealer*, not a fairness *guarantee*.** My model hit 93.3% accuracy and I felt fine about it for exactly two seconds, until I broke the number down by role and saw a 75% FNR for Operators.

Three specific takeaways that change how I think about FIELDPROOF:

1. **Auditing must be baked into the pipeline, not bolted on afterward.** If I had only reported overall accuracy in a final project demo, I would have made a model that quietly punishes a specific worker role look like a success. Every future FIELDPROOF model I build should have per-subgroup FPR/FNR dashboards from the first training run, not as an afterthought.

2. **Proxy features are the real enemy.** Removing `worker_role` did almost nothing because the sensor streams themselves encoded role information. For real FIELDPROOF deployment this means I cannot rely on "we don't collect the protected attribute" as a defense — I have to prove via subgroup audits that the model is fair, using the protected attribute *as an evaluation variable* even when it is excluded from training.

3. **Sample size matters for fairness, not just accuracy.** Some of my subgroup numbers (Pipeline-West n=21, Helmet Sensor n=21) are suspicious precisely because they are small. A "fairness failure" in a tiny subgroup might just be sampling noise; a real audit needs confidence intervals around each subgroup metric, not point estimates. This is the next thing I would add.

For FIELDPROOF specifically: before any pilot deployment in a real workplace, I need subject-independent splits (train on one set of workers, test on another), confidence intervals on every subgroup metric, and a constraint in the training objective that explicitly penalizes between-group FNR gaps. Without those, the model is not ready to affect a real worker's paycheck or safety.

---

**End of notebook — Nestor Villalobos, ITAI 1371, Spring 2026**